<center><img src="../../image/dice.png" alt="DICE" style="width: 300px;"/></center>

# 11.摄像头标定

@－－－－DICE－－－－<br>
@  文件名：11.摄像头标定.py <br>
@  版本：V2.0 <br>
@  说明：我们第一步需要开启摄像头作为显示
我们来学习一下摄像头的开启方式之一吧，一般来说，对于我们使用摄像头的时候，首先我们需要测试我们的摄像头是否正常工作，画面是否清晰，让我们一起来测试我们的摄像头吧！！！<br>

我们开始测试摄像头是否可用

### 调整机械臂到达Home 点，便于获取图像

In [1]:
import math
from arm5dof_uservo import Arm5DoFUServo
from config import *

### 机械臂创建

In [2]:
arm = Arm5DoFUServo()

### 到达Home点

In [3]:
# 回到Home点
arm.home()

In [4]:
#到达拍摄位置
arm.uservo.set_servo_angle(2,-60,interval=1000)
arm.uservo.set_servo_angle(3,-95,interval=1000)
arm.uservo.set_servo_angle(5,90,interval=1000)
arm.uservo.wait()

### 载入对应的库文件

In [5]:
from camera import Camera
import cv2
import time
import numpy as np
import pickle
import subprocess
import math
import yaml
import logging

捕获一帧图片

In [6]:
def main(argv):
    global button_flag
    import opencv_jupyter_ui as jcv2
    '''调整相机参数, 预览图像'''
    img_cnt = FLAGS.img_cnt
    # 是否显示FPS信息
    is_show_fps = True #False
    # 创建相机对象
    camera = Camera()
    # 初始相机
    capture = camera.capture
    print("FLAGS.rm_distortion = {}".format(FLAGS.rm_distortion))

    if FLAGS.rm_distortion:
        # 载入标定数据
        camera.load_cam_calib_data()
    jcv2.setKeys(['q','s','enter','space'])
    fps = 30 # 设定一个初始值
    while True:
        start = time.time()
        # 获取彩图与深度图
        ret,image = camera.read()
        if not ret:
            logging.error('图像获取失败')
            break
        if FLAGS.rm_distortion:
            # 图像去除畸变
            image = camera.remove_distortion(image)

        # 创建画布
        canvas = np.copy(image)
        if is_show_fps:
            # 添加帮助信息
            cv2.putText(canvas, text='S:Save Image',\
                    org=(50, camera.config['img_height']-100), fontFace=cv2.FONT_HERSHEY_SIMPLEX, \
                    fontScale=1, thickness=2, lineType=cv2.LINE_AA, color=(0, 0, 255))
            cv2.putText(canvas, text='Q: Quit',\
                    org=(50, camera.config['img_height']-50), fontFace=cv2.FONT_HERSHEY_SIMPLEX, \
                    fontScale=1, thickness=2, lineType=cv2.LINE_AA, color=(0, 0, 255))

            # 在画布上添加帧率的信息
            cv2.putText(canvas, text='FPS: {}'.format(fps),\
                    org=(50, 50), fontFace=cv2.FONT_HERSHEY_SIMPLEX, \
                    fontScale=1, thickness=2, lineType=cv2.LINE_AA, color=(0, 0, 255))
        # 更新窗口“color”中的图片
        #jcv2.imshow('color',canvas,width=640,height=480)
        img.value = bgr8_to_jpeg(canvas)

        end = time.time()
        fps = int(0.6*fps +  0.4*1/(end-start))
                   
        key = jcv2.waitKey(1)
        if key == ord('q'):
            # 如果按键为q 代表quit 退出程序
            break
        elif key == ord('s'):
            # s键代表保存数据
            cv2.imwrite('{}/{}.png'.format(FLAGS.img_path, img_cnt), image)
            logging.info("截图，并保存在  {}/{}.png".format(FLAGS.img_path, img_cnt))
            img_cnt += 1
        
    # 关闭摄像头
    capture.release()
    jcv2.destroyAllWindows()

In [7]:
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg


img = widgets.Image(format='jpeg', width=640, height=480)
display(img)


if __name__ == '__main__':
    import logging
    import sys
    sys.argv = " --img_path data/caliboard".split(" ")
    from absl import app
    from absl import flags

    # 设置日志等级
    logging.basicConfig(level=logging.INFO)
    
    # 定义参数
    FLAGS = flags.FLAGS
    flags.DEFINE_integer('img_cnt', 0, '图像计数的起始数值')
    flags.DEFINE_string('img_path', 'data/image_raw', '图像的保存地址')
    flags.DEFINE_boolean('rm_distortion', False, '载入相机标定数据, 去除图像畸变')

    # 运行主程序
    app.run(main)

Image(value=b'', format='jpeg', height='480', width='640')

FLAGS.rm_distortion = False


I0423 13:51:59.240572 547809808400 246703818.py:59] 截图，并保存在  data/caliboard/0.png
I0423 13:52:05.563251 547809808400 246703818.py:59] 截图，并保存在  data/caliboard/1.png
I0423 13:52:09.412363 547809808400 246703818.py:59] 截图，并保存在  data/caliboard/2.png
I0423 13:54:08.809024 547809808400 246703818.py:59] 截图，并保存在  data/caliboard/3.png
I0423 13:54:17.204284 547809808400 246703818.py:59] 截图，并保存在  data/caliboard/4.png
I0423 13:54:21.687387 547809808400 246703818.py:59] 截图，并保存在  data/caliboard/5.png
I0423 13:54:26.162111 547809808400 246703818.py:59] 截图，并保存在  data/caliboard/6.png
I0423 13:54:30.885030 547809808400 246703818.py:59] 截图，并保存在  data/caliboard/7.png
I0423 13:54:39.165483 547809808400 246703818.py:59] 截图，并保存在  data/caliboard/8.png
I0423 13:54:43.091628 547809808400 246703818.py:59] 截图，并保存在  data/caliboard/9.png
I0423 13:54:58.852684 547809808400 246703818.py:59] 截图，并保存在  data/caliboard/10.png
I0423 13:55:07.165174 547809808400 246703818.py:59] 截图，并保存在  data/caliboard/11.png
I0423 13:55:10

SystemExit: 

/usr/local/lib/python3.8/dist-packages/IPython/core/interactiveshell.py:3406: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### 运行此脚本完成相机标定

In [8]:
! python3 camera_calibration.py

检测到30张标定图像
相机内参 intrinsic
[[710.261   0.    287.298]
 [  0.    754.919 225.602]
 [  0.      0.      1.   ]]
畸变参数 distortion
[[-0.234  0.1   -0.01  -0.004 -0.299]]
